In [3]:
import pandas as pd
import numpy as np

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
hist_df = pd.read_csv(f'{base_path}steam_indie_review_histogram.csv')

df.columns = df.columns.str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days
df['days_since_release'] = df['days_since_release'].clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

df['tag_list'] = df['tags'].str.split(',').apply(lambda x: [t.strip() for t in x] if isinstance(x, list) else [])
tag_exploded = df.explode('tag_list')

tag_analysis = tag_exploded.groupby('tag_list').agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

result = tag_analysis[tag_analysis['game_count'] >= 20].sort_values('velocity', ascending=False).head(20)
print(result)

                     velocity   sentiment  value_score  stability  game_count
tag_list                                                                     
"Indie": 25          0.372700  457.260908     0.047988   9.343768          22
"Singleplayer": 97   0.319709  435.670883     0.000000  11.043216          25
"Indie": 24          0.244884  467.656875     0.016135  13.654196          26
"Singleplayer": 105  0.218065  388.314287     0.006550  10.583761          26
"Indie": 79          0.171580  389.662587     0.000000  11.338187          24
"Singleplayer": 82   0.166379  387.047659     0.006845  15.351580          28
"Singleplayer": 98   0.166234  367.554650     0.000000  10.629948          32
"Singleplayer": 130  0.160451  363.906770     0.007890   8.579481          20
"Indie": 23          0.159463  406.726424     0.009415  10.857705         100
"Singleplayer": 104  0.151432  340.494359     0.007520   6.497677          20
"Casual": 106        0.145121  390.945657     0.000000   9.62501

In [ ]:
import pandas as pd
import numpy as np
import ast

# 1. 데이터 로드
base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

# 2.4대 시그널 지표 계산
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

# 3. 팀원분의 카테고리별 태그 리스트
mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

def extract_tags(tag_str, target_list):
    try:
        tags_dict = ast.literal_eval(tag_str)
        return [t for t in tags_dict.keys() if t in target_list]
    except:
        return []

# 4. 태그 추출 및 카테고리별 분석
categories = {
    'Mechanics': mechanics,
    'Themes': themes,
    'Moods': moods,
    'Visuals': visuals
}

for cat_name, tag_list in categories.items():
    # 열 생성 및 Explode
    df['temp_tags'] = df['tags'].apply(lambda x: extract_tags(x, tag_list))
    exploded_df = df.explode('temp_tags').dropna(subset=['temp_tags'])
    
    # 지표 산출 (표본 50개 이상)
    cat_analysis = exploded_df.groupby('temp_tags').agg({
        'velocity': 'median',
        'sentiment': 'median',
        'value_score': 'median',
        'stability': 'median',
        'appid': 'count'
    }).rename(columns={'appid': 'game_count'})
    
    result = cat_analysis[cat_analysis['game_count'] >= 50].sort_values('velocity', ascending=False).head(5)
    
    print(f"\n=== Top 5 Tags in {cat_name.upper()} (by Velocity) ===")
    print(result)


=== Top 5 Tags in MECHANICS (by Velocity) ===
              velocity   sentiment  value_score  stability  game_count
temp_tags                                                             
Turn-Based    0.262731  417.989898      0.00501   5.685372         174
Online Co-Op  0.185464  371.740411      0.00000   8.318560         522
Sandbox       0.155157  361.266480      0.00000   7.341907         775
Co-op         0.128360  353.934701      0.00000   8.189263         657
Crafting      0.118146  334.264674      0.00000   6.932709         572

=== Top 5 Tags in THEMES (by Velocity) ===
                  velocity   sentiment  value_score  stability  game_count
temp_tags                                                                 
Historical        0.103596  352.501636          0.0   8.225611         304
Post-apocalyptic  0.101049  331.685863          0.0   8.809515         402
Anime             0.093459  356.409408          0.0   9.602328        1070
Military          0.086238  337.30189

In [5]:
import pandas as pd
import numpy as np
import ast
from itertools import combinations

# 1. 데이터 로드
base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

# 2. 4대 시그널 계산
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

# 3. 유효 태그 필터링 세팅
mechanics = ['Singleplayer', 'Multiplayer', 'Co-op', 'PvP', 'Online Co-Op', 'Local Co-Op', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

valid_tags = set(mechanics + themes + moods + visuals)

def extract_valid_tags(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        # 조합 시 순서가 바뀌어 중복되는 것을 막기 위해 sorted 사용
        return sorted([t for t in tags_dict.keys() if t in valid_tags])
    except:
        return []

df['valid_tags'] = df['tags'].apply(extract_valid_tags)

# 4. 태그 조합 생성
def get_combinations(tag_list):
    if len(tag_list) < 2:
        return []
    return list(combinations(tag_list, 2))

df['tag_pairs'] = df['valid_tags'].apply(get_combinations)

# 5. Explode 및 4대 시그널 집계
exploded_df = df.explode('tag_pairs').dropna(subset=['tag_pairs'])

synergy_analysis = exploded_df.groupby('tag_pairs').agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

# 6. 표본 30개 이상 필터링 및 컬럼 분리
synergy_results = synergy_analysis[synergy_analysis['game_count'] >= 30].reset_index()

synergy_results['tag_1'] = synergy_results['tag_pairs'].apply(lambda x: x[0])
synergy_results['tag_2'] = synergy_results['tag_pairs'].apply(lambda x: x[1])
synergy_results = synergy_results.drop(columns=['tag_pairs'])

# 가독성을 위해 컬럼 순서 재배치
synergy_results = synergy_results[['tag_1', 'tag_2', 'game_count', 'velocity', 'sentiment', 'value_score', 'stability']]

# 7. 충성도(Value Score) 기준 상위 20개 조합 출력
print("=== 2단계: 태그 시너지 분석 결과 (Value Score 기준 Top 20) ===")
print(synergy_results.sort_values('value_score', ascending=False).head(20).to_string(index=False))

# (필요 시 'stability'나 'velocity' 기준으로도 sort_values를 변경하여 확인 가능함)

=== 2단계: 태그 시너지 분석 결과 (Value Score 기준 Top 20) ===
           tag_1        tag_2  game_count  velocity  sentiment  value_score  stability
        Crafting   Story Rich          81  0.319579 445.227564     0.025639   5.136267
      Souls-like   Story Rich          64  0.188717 413.158575     0.024705   6.026166
      Historical      Sandbox          33  0.658730 385.268581     0.024649   4.441328
    Online Co-Op      Sandbox          75  0.572071 460.485392     0.024132   5.579866
         Sandbox   Story Rich          70  0.318293 422.225349     0.023994   5.208016
           Co-op         Gore          33  0.344768 418.442504     0.022500   6.916540
     Multiplayer   Open World         112  0.569767 417.016536     0.020394   5.295376
           Co-op      Sandbox          78  0.836957 491.802268     0.020204   6.841492
    Online Co-Op   Open World          56  0.677049 448.625778     0.020204   5.100304
        Crafting Online Co-Op          51  0.708872 436.105757     0.019560   4.

In [6]:
import pandas as pd
import numpy as np
import ast

# 1. 데이터 로드 및 지표 계산 (준환님 수식 적용)
base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

# 2. 팀원 카테고리 정의
mechanics = ['Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Strategy'] # 핵심 위주로 압축
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Anime', 'Historical']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics']

# 3. 게임별 '대표 속성 세트' 추출 (카테고리당 1개씩 대표값만 추출하여 조합 단순화)
def get_core_mix(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        tags = set(tags_dict.keys())
        
        m = [t for t in mechanics if t in tags][:1] # 메커니즘 1개
        t = [t for t in themes if t in tags][:1]    # 테마 1개
        v = [t for t in visuals if t in tags][:1]   # 비주얼 1개
        
        core_mix = sorted(m + t + v)
        return " + ".join(core_mix) if len(core_mix) >= 2 else None
    except:
        return None

df['core_combination'] = df['tags'].apply(get_core_mix)

# 4. 조합별 4대 시그널 집계
combination_analysis = df.dropna(subset=['core_combination']).groupby('core_combination').agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

# 5. 표본 30개 이상 필터링
final_combinations = combination_analysis[combination_analysis['game_count'] >= 30].sort_values('velocity', ascending=False)

print("### 2단계 확장: 핵심 카테고리 조합(N개) 시그널 분석")
print(final_combinations.head(20))

### 2단계 확장: 핵심 카테고리 조합(N개) 시그널 분석
                            velocity   sentiment  value_score  stability  \
core_combination                                                           
Fantasy + Strategy          0.182065  378.180255     0.003169   6.641288   
2D + Fantasy + Turn-Based   0.170611  393.144796     0.000000   6.066121   
2D + Turn-Based             0.169492  401.384496     0.020026   7.841715   
2D + Sci-fi + Survival      0.129787  375.426189     0.000000   9.037820   
2D + Horror + Puzzle        0.104400  374.355933     0.000000  10.874131   
2D + Anime + Strategy       0.096459  350.065886     0.000000   9.468928   
2D + Fantasy + Open World   0.094484  375.234860     0.000000   7.466570   
2D + Anime                  0.092348  343.754134     0.000000  10.855043   
Horror + Puzzle             0.090909  331.076377     0.000000  10.293868   
3D + Open World             0.090517  288.804356     0.000000   8.008008   
2D + Open World             0.087719  349.925815     0

In [8]:
import pandas as pd
import numpy as np
import ast
from itertools import combinations

base_path = '../../../../data/preprocessed/'
df = pd.read_csv(f'{base_path}steam_indie_games_graded.csv')
df.columns = df.columns.str.strip()

# 4대 시그널 계산
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
current_date = pd.to_datetime('2026-05-06')
df['days_since_release'] = (current_date - df['release_date']).dt.days.clip(lower=1)

df['velocity'] = df['total_reviews'] / df['days_since_release']
df['sentiment'] = df['positive_rate'] * np.log1p(df['total_reviews'])
df['value_score'] = df['recommendations_total'].fillna(0) / (df['owners_lower'] + 1)
df['stability'] = df['positive_rate'] / (df['price'] + 1)

# 카테고리 정의
mechanics = ['Singleplayer', 'Multiplayer', 'Roguelike', 'Turn-Based', 'Open World', 'Survival', 'Puzzle', 'Platformer', 'Metroidvania', 'Souls-like', 'FPS', 'Tactical', 'Dungeon Crawler', 'Sandbox', 'Crafting', 'Simulation', 'Strategy', 'Action-Adventure', 'Bullet Hell', 'Hack and Slash']
themes = ['Sci-fi', 'Fantasy', 'Horror', 'Historical', 'Cyberpunk', 'Post-apocalyptic', 'Space', 'Medieval', 'Steampunk', 'Zombies', 'Magic', 'War', 'Mystery', 'Lovecraftian', 'Comedy', 'Cute', 'Nature', 'Anime', 'Military']
moods = ['Atmospheric', 'Relaxing', 'Dark', 'Funny', 'Psychological Horror', 'Difficult', 'Casual', 'Emotional', 'Stylized', 'Minimalist', 'Violent', 'Gore', 'Colorful', 'Beautiful', 'Story Rich', 'Surreal']
visuals = ['2D', '3D', 'Pixel Art', 'Pixel Graphics', 'Low-Poly', 'Voxel', 'Hand-drawn', 'Anime', 'Cartoony', 'Realistic', 'Isometric', 'Top-Down', 'Side Scroller', 'First-Person', 'Third Person']

valid_tags = set(mechanics + themes + moods + visuals)

def get_tag_pairs(tag_str):
    try:
        tags_dict = ast.literal_eval(tag_str)
        core_tags = sorted([t for t in tags_dict.keys() if t in valid_tags])
        return list(combinations(core_tags, 2))
    except:
        return []

# 타겟 등급 필터링 및 조합 생성
target_grades = ['high_high', 'high_mid', 'mid_high']
df_target = df[df['performance_grade'].isin(target_grades)].copy()
df_target['tag_pairs'] = df_target['tags'].apply(get_tag_pairs)

# 조합별 집계
exploded = df_target.explode('tag_pairs').dropna(subset=['tag_pairs'])

# 등급별 + 조합별 통계 산출
grade_pair_analysis = exploded.groupby(['performance_grade', 'tag_pairs']).agg({
    'velocity': 'median',
    'sentiment': 'median',
    'value_score': 'median',
    'stability': 'median',
    'appid': 'count'
}).rename(columns={'appid': 'game_count'})

# 등급별 상위 10개 조합 출력
for grade in target_grades:
    print(f"\n### [{grade.upper()}] TOP 10 SYNERGY RECIPES ###")
    res = grade_pair_analysis.loc[grade].sort_values('game_count', ascending=False).head(10)
    print(res)


### [HIGH_HIGH] TOP 10 SYNERGY RECIPES ###
                                velocity   sentiment  value_score  stability  \
tag_pairs                                                                      
(2D, Singleplayer)              1.862408  660.708920     0.031909   7.326675   
(Casual, Singleplayer)          2.175703  672.125644     0.033650   8.677098   
(Simulation, Singleplayer)      2.747786  673.567016     0.035348   7.512830   
(Singleplayer, Story Rich)      2.081510  670.496569     0.035198   6.061391   
(Atmospheric, Singleplayer)     2.057307  667.783231     0.034823   7.007362   
(3D, Singleplayer)              2.015284  663.192246     0.033548   7.873385   
(Pixel Graphics, Singleplayer)  2.114079  669.286694     0.032358   7.265901   
(Singleplayer, Strategy)        2.671515  659.037703     0.028197   6.052780   
(First-Person, Singleplayer)    2.504043  668.310824     0.037901   7.089795   
(2D, Pixel Graphics)            2.078796  673.974592     0.030380   7.506327